# Multivariate Regression

## 1. Data loading and vetting

In [3]:
import pandas as pd

# African countries, spelled exactly as they appear in the survey's Country column
AFRICA = {
    "Algeria", "Angola", "Benin", "Botswana", "Burkina Faso", "Burundi", "Cameroon",
    "Cape Verde", "Central African Republic", "Chad", "Congo, Republic of the...",
    "Côte d'Ivoire", "Democratic Republic of the Congo", "Djibouti", "Egypt",
    "Equatorial Guinea", "Ethiopia", "Gabon", "Gambia", "Ghana", "Guinea",
    "Guinea-Bissau", "Kenya", "Lesotho", "Liberia", "Libyan Arab Jamahiriya",
    "Madagascar", "Malawi", "Mali", "Mauritania", "Mauritius", "Morocco", "Mozambique",
    "Namibia", "Niger", "Nigeria", "Rwanda", "Senegal", "Sierra Leone", "Somalia",
    "South Africa", "Sudan", "Swaziland", "Togo", "Tunisia", "Uganda",
    "United Republic of Tanzania", "Zambia", "Zimbabwe",
}
YEARS = (2023, 2024, 2025, 2026)
YOUTH = ["18-24 years old", "25-34 years old"]   # AU youth definition: up to 35

frames = {}
for year in YEARS:
    d = pd.read_csv(f"stackoverflow_{year}.csv", low_memory=False)
    d["SurveyYear"] = year
    africa_pay = d[d["Country"].isin(AFRICA) & d["ConvertedCompYearly"].notna()]
    frames[year] = africa_pay
    print(f"{year}: all={len(d):>6} | African={d['Country'].isin(AFRICA).sum():>5} "
          f"| African with pay={len(africa_pay):>4} | columns={d.shape[1]}")

df = pd.concat(frames.values(), ignore_index=True)
pay = df["ConvertedCompYearly"]

print("\nCombined African rows with pay:", len(df))
print("Of which youth (18–34):", df["Age"].isin(YOUTH).sum())
print("\nPay (USD/year):\n", pay.describe(percentiles=[.01, .05, .5, .95, .99]).round(0))
print("\nTop countries:\n", df["Country"].value_counts().head(8))

# Columns renamed between years
for old, new in [("EdLevel", "Education"), ("RemoteWork", "WorkSetting"), ("DevType", "Role")]:
    print(f"\n{old}/{new} present per year:",
          {y: (old in f.columns, new in f.columns) for y, f in frames.items()})

2023: all= 89184 | African= 2534 | African with pay= 926 | columns=85
2024: all= 65437 | African= 1837 | African with pay= 555 | columns=115
2025: all= 49191 | African= 1178 | African with pay= 645 | columns=173
2026: all= 30903 | African=  717 | African with pay= 383 | columns=213

Combined African rows with pay: 2509
Of which youth (18–34): 1751

Pay (USD/year):
 count        2509.0
mean       109490.0
std       1783547.0
min             1.0
1%              6.0
5%            133.0
50%         13598.0
95%        103743.0
99%        270476.0
max      73607918.0
Name: ConvertedCompYearly, dtype: float64

Top countries:
 Country
South Africa    789
Nigeria         425
Egypt           233
Kenya           231
Morocco         108
Tunisia          89
Ghana            79
Ethiopia         73
Name: count, dtype: int64

EdLevel/Education present per year: {2023: (True, False), 2024: (True, False), 2025: (True, False), 2026: (False, True)}

RemoteWork/WorkSetting present per year: {2023: (True, F

No single survey year has 1,000 African respondents with pay (926 / 555 / 645 / 383), so I combine 2023–2026 (2,509 rows). Pay ranges from $1 to $73.6M against a median of about $13.6k, so implausible values are removed in the next step.

## 2. Data cleaning and harmonisation

In [4]:
import numpy as np
import pandas as pd

# Columns that were renamed between survey years 
RENAME = {"EdLevel": "Education", "RemoteWork": "WorkSetting", "DevType": "Role"}


def years_to_num(s):
    """'Less than 1 year' → 0.5, 'More than 50 years' → 51, numbers → float."""
    return pd.to_numeric(
        s.replace({"Less than 1 year": 0.5, "More than 50 years": 51}), errors="coerce"
    )


def education(x):
    x = str(x).lower()
    if "primary" in x or "secondary" in x:  return "Secondary or less"
    if "without earning" in x:              return "Some university, no degree"
    if "associate" in x:                    return "Associate degree"
    if "bachelor" in x:                     return "Bachelor's degree"
    if "master" in x:                       return "Master's degree"
    if "professional" in x:                 return "Doctorate or professional degree"
    return np.nan                        


def work_mode(x):
    x = str(x).lower()
    if x.startswith("remote"):    return "Remote"
    if x.startswith("in-person"): return "In-person"
    if x.startswith("hybrid") or x.startswith("your choice"): return "Hybrid"
    return np.nan


def org_size(x):
    x = str(x)
    if x.startswith("Just me"):                                  return "Freelancer (just me)"
    if x.startswith(("2 to 9", "10 to 19", "Less than 20")):    return "Under 20"
    if x.startswith("20 to 99"):                                 return "20-99"
    if x.startswith("100 to 499"):                               return "100-499"
    if x.startswith(("500 to 999", "1,000 to 4,999")):          return "500-4,999"
    if x.startswith(("5,000 to 9,999", "10,000 or more")):      return "5,000+"
    return np.nan                                                


def employment(x):
    x = str(x)
    employed, freelance = "Employed" in x, "Independent contractor" in x
    if employed and freelance: return "Employed and freelancing"
    if freelance:              return "Freelancer or self-employed"
    if employed:               return "Employed"
    return np.nan                                                


def role_group(x):
    x = str(x).lower()
    if "full-stack" in x:  return "Full-stack developer"
    if "back-end" in x:    return "Back-end developer"
    if "front-end" in x:   return "Front-end developer"
    if "mobile" in x:      return "Mobile developer"
    if any(k in x for k in ("data", "machine learning", "ai/ml")): return "Data or AI"
    if x == "nan":         return np.nan
    return "Other tech role"


frames = []
for year in YEARS:
    d = pd.read_csv(f"stackoverflow_{year}.csv", low_memory=False).rename(columns=RENAME)
    d = d[d["Country"].isin(AFRICA) & d["ConvertedCompYearly"].notna()].copy()

    # Learning pathway: 2023–2025 use LearnCode; 2026 moved the answers to LearnCodeHow
    learn = d["LearnCodeHow"] if year == 2026 else d["LearnCode"]
    d["OnlineCourses"] = learn.fillna("").str.contains("online course", case=False).astype(int)

    d["WorkExp"] = years_to_num(d["WorkExp"])
    if "YearsCodePro" in d:
        d["WorkExp"] = d["WorkExp"].fillna(years_to_num(d["YearsCodePro"]))

    d["SurveyYear"] = year
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)

clean = pd.DataFrame({
    "Country":        raw["Country"],
    "AgeGroup":       raw["Age"],
    "Education":      raw["Education"].map(education),
    "YearsCode":      years_to_num(raw["YearsCode"]),
    "WorkExp":        raw["WorkExp"],
    "WorkMode":       raw["WorkSetting"].map(work_mode),
    "OrgSize":        raw["OrgSize"].map(org_size),
    "EmploymentType": raw["Employment"].map(employment),
    "RoleGroup":      raw["Role"].map(role_group),
    "OnlineCourses":  raw["OnlineCourses"],
    "SurveyYear":     raw["SurveyYear"],
    "AnnualPayUSD":   raw["ConvertedCompYearly"],
})


clean = clean[clean["AgeGroup"].isin(["18-24 years old", "25-34 years old"])]
clean["AgeGroup"] = clean["AgeGroup"].str.replace(" years old", "", regex=False)

# Remove implausible pay (under $50/month or over $300k/year)
clean = clean[clean["AnnualPayUSD"].between(600, 300_000)]

before = len(clean)
clean = clean.dropna().reset_index(drop=True)

# Countries with few respondents (<30) are grouped into "Other African country"
counts = clean["Country"].value_counts()
clean["Country"] = clean["Country"].where(clean["Country"].map(counts) >= 30, "Other African country")

clean[["YearsCode", "WorkExp"]] = clean[["YearsCode", "WorkExp"]].astype(float)

print(f"Rows after youth + pay filters: {before} → after dropping missing values: {len(clean)}")
print("Features:", clean.shape[1] - 1, "| Target: AnnualPayUSD")
print(clean.dtypes, "\n")
for c in ["Country", "Education", "WorkMode", "OrgSize", "EmploymentType", "RoleGroup"]:
    print(c, clean[c].value_counts().to_dict(), "\n")

clean.to_csv("data.csv", index=False)
print("Saved data.csv:", clean.shape)
clean.head()

Rows after youth + pay filters: 1518 → after dropping missing values: 1363
Features: 11 | Target: AnnualPayUSD
Country               str
AgeGroup              str
Education             str
YearsCode         float64
WorkExp           float64
WorkMode              str
OrgSize               str
EmploymentType        str
RoleGroup             str
OnlineCourses       int64
SurveyYear          int64
AnnualPayUSD      float64
dtype: object 

Country {'South Africa': 333, 'Other African country': 246, 'Nigeria': 223, 'Egypt': 150, 'Kenya': 145, 'Morocco': 80, 'Tunisia': 58, 'Ghana': 52, 'Ethiopia': 40, 'Algeria': 36} 

Education {"Bachelor's degree": 834, "Master's degree": 203, 'Some university, no degree': 199, 'Secondary or less': 79, 'Associate degree': 34, 'Doctorate or professional degree': 14} 

WorkMode {'Remote': 540, 'Hybrid': 520, 'In-person': 303} 

OrgSize {'Under 20': 489, '20-99': 347, '100-499': 183, '500-4,999': 154, 'Freelancer (just me)': 112, '5,000+': 78} 

EmploymentType 

,Country,AgeGroup,Education,YearsCode,WorkExp,WorkMode,OrgSize,EmploymentType,RoleGroup,OnlineCourses,SurveyYear,AnnualPayUSD
0,Nigeria,25-34,Bachelor's degree,8.0,6.0,Remote,20-99,Employed and freelancing,Full-stack developer,1,2023,120000.0
1,Egypt,25-34,Master's degree,5.0,5.0,Hybrid,"500-4,999",Employed,Other tech role,1,2023,7000.0
2,Morocco,25-34,Bachelor's degree,5.0,3.0,Hybrid,Freelancer (just me),Employed,Full-stack developer,0,2023,12851.0
3,Egypt,18-24,Bachelor's degree,4.0,2.0,Remote,Under 20,Employed,Back-end developer,1,2023,3247.0
4,Other African country,18-24,Bachelor's degree,4.0,1.0,In-person,Under 20,Employed,Full-stack developer,1,2023,5715.0


I added together the survey results for each year, renamed the columns for uniformity, and retained the Africans who were age 18–34 and earning between $600 and $300,000. I combined all the text answers, added an online-course factor, converted experience to numbers, with missing experience rows being filled with the number zero, and omitted rows where the person had no experience or experience that was not possible for their age. There were 1355 rows in data.csv after grouping African countries with less than 30 responses as ‘Other African country'.